# RecoverKv on LongGenBench: long-generation accuracy (branch `Clustered_int2_paused_fast`)

**What this measures.** LongGenBench (Liu et al., EMNLP'24 Findings) as **SCOPE** (Wu et al., ACL'25) packages and
runs it for Llama-3.1-8B-Instruct. K questions from GSM8K, MMLU or CommonsenseQA go into one prompt, and the model
must answer all of them, in order, in one long greedy generation. The **8K** setting asks for 60 / 60 / 80 answers in
up to 7,950 new tokens; the prompts are only ~4–6K tokens (a few MMLU subjects are longer), so almost every token is
decoded from the compressed cache (the cache compresses the prompt on decode step 0).

At your settings and a 5,000-token prompt, the cache resolves to about 870 fp16 keys plus about 820 int2 windows
(7,400 keys in all; arithmetic from `resolve()`, step 10 prints the exact figures for the real prompts). So the prompt
is int2-compressed on the first decode step, the read gate selects over it from then on, and keys start being dropped
roughly 2,400 generated tokens in.

Two arms see byte-identical prompts:
* **fullkv**: HF `DynamicCache`, flash-attention-2. This is SCOPE's `ALLKV`.
* **ours**: the windowed two-tier cache at your settings (step 8), flash backend, read gate on.
* *(optional)* **ctrl_gate1.0**: ours at `quant_gate_ratio = 1.0`, the gate's control arm.

**Why a number from this notebook can be trusted.** Each row is checked, not assumed:

| claim | how it is checked |
|---|---|
| the data is SCOPE's | SHA-256 of every file against the pin (`Linking-ai/SCOPE` @ `8cbae48`) |
| the prompt is the string SCOPE tokenized | every prompt SCOPE recorded in its own output is rebuilt byte for byte (step 6) |
| the metric is SCOPE's metric | re-scoring SCOPE's own full-cache predictions gives back its published 44.44 / 51.01 / 64.92 (8K) and 53.26 / 54.40 / 71.67 (4K), exactly (step 6) |
| the decoding is SCOPE's | greedy, `min_length = prompt + 1`, EOS = the `eot_id` token (128009), B = 1, fp16, flash-attn 2, one BOS: recorded in every sidecar and re-checked |
| **ours ran the gated method** | `read_gate.verdict == "gated"` with `gated == fired > 0` in every part, plus the realised read fraction |
| ours is at your operating point | every cache knob is re-read from the sidecar and compared with step 8, including the resolved geometry |
| the cache was really compressed | per example: whether the int2 tier was used, whether keys were dropped, and the KV-byte fraction held |
| fullkv reproduces SCOPE | accuracy against SCOPE's `ALLKV` on the same examples, and how long the two generations agree character for character |

**Read three numbers, not one.** `acc` is SCOPE's metric. It pairs answers with questions **by position**, so one skipped
answer shifts every later answer onto the wrong gold. `aligned` pairs `Answer_N` with question N using the same answer
pattern. `coverage` is the share of questions that got an `Answer_N` block at all; a cache that loses its place in a long
answer shows up there first. If `acc` and `aligned` move differently between arms, read the per-example records before
quoting either.

Run the cells in order, from step 1. Each **part** (one task × one chunk of examples × one arm) is verified and copied to
Google Drive as it finishes. If you run the notebook again, parts that are already complete and verified (same settings,
commit and GPU model) are reused instead of re-run.

**Before step 1:**
* *Runtime → Change runtime type → A100 GPU.*
* Accept the licence at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct, then add your HF token as a Colab
  secret named `HF_TOKEN` (key icon in the left bar) and allow this notebook to read it.

### How long it takes (8K setting, one A100)
These are arithmetic estimates, not measurements. They use SCOPE's own full-cache answer lengths and assume about 30 ms
per token for fullkv and about 55 ms per token for ours at B = 1. After each part, step 11 re-projects the rest from the
speed it actually measured.

| task | examples | new tokens per arm | fullkv | ours |
|---|---|---|---|---|
| gsm8k | 21 | ~125–145K | ~1.0–1.2 h | ~1.9–2.2 h |
| mmlu | 53 | ~250–285K | ~2.1–2.4 h | ~3.8–4.4 h |
| csqa | 15 | ~110K (almost all reach 7,950) | ~0.9 h | ~1.6–1.7 h |
| **all** | 89 | | **~4–4.5 h** | **~7.3–8.3 h** |

The 4K setting (`SETTING = "4K"`) costs about the same in total: it has more examples, each with a shorter answer.
`NUM_EXAMPLES` or `TASKS` in step 8 shortens a first pass. A Colab disconnect costs at most the part that was running.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

## 2. Clone the branch, pinned to the commit that adds the LongGenBench runner
Your efficiency notebook pins `7beb3da`. This one pins `72d8e07`, the first commit with `mode: longgenbench`. Between
the two, one commit touched the cache code (`db9a7ed`): it adds the completion-ablation knob, which is off by default
(`quant_completion: null` is the shipped kernel path), and an output-dtype option in the decode kernel that only an
ablation arm sets. Every "ours" part is checked for the default: `quant_completion.mode == "native"` and
`read_gate.completion_applied == 0`. Strictly, though, the speed table and this accuracy table come from adjacent
commits, so say so if you quote them side by side, or re-run the efficiency notebook at `72d8e07`.

In [ ]:
%cd /content
!rm -rf RecoverKv
!git clone -q --branch Clustered_int2_paused_fast --single-branch https://github.com/shaswata965-coder/RecoverKv.git
%cd /content/RecoverKv
!git checkout -q 72d8e07a93b36536d9a47f07bebb40cea66fed32 && git log -1 --format='%h %s'
!test -f modules/evaluation/longgenbench_runner.py && echo "LongGenBench runner present"

## 3. Install the pinned stack (~5 min)
torch 2.6 is the build the recorded GPU runs used; transformers must be exactly 4.47.1
(`utils/cache_factory.py` refuses anything newer). flash-attn comes from its prebuilt wheel. If pip
starts *compiling* flash-attn, stop the cell, because that means no wheel matched this Python.

**When this cell finishes: Runtime → Restart session**, then continue at step 4.

In [ ]:
%%bash
set -euo pipefail
pip -q install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

# flash-attn: prebuilt wheel matching this python / torch 2.6 / C++ ABI
read PYV ABI < <(python -c "import sys,torch; print(f'cp{sys.version_info[0]}{sys.version_info[1]}', 'TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE')")
ok=0
for V in 2.7.4.post1 2.8.3; do
  WHL="https://github.com/Dao-AILab/flash-attention/releases/download/v${V}/flash_attn-${V}+cu12torch2.6cxx11abi${ABI}-${PYV}-${PYV}-linux_x86_64.whl"
  echo "trying $WHL"
  if pip -q install --no-deps "$WHL"; then ok=1; break; fi
done
[ "$ok" = 1 ] || { echo "no prebuilt flash-attn wheel for ${PYV}/torch2.6/abi${ABI}"; exit 1; }

pip -q install "transformers==4.47.1" "tokenizers>=0.21,<0.22" "huggingface_hub>=0.24,<1.0" \
               "datasets>=2.16" "accelerate>=0.26" sentencepiece protobuf pyyaml einops pytest
python -c "import torch, transformers, flash_attn; print('torch', torch.__version__, '| transformers', transformers.__version__, '| flash_attn', flash_attn.__version__)"
echo "NOW: Runtime -> Restart session, then run step 4."

## 4. Hugging Face login + model download (~16 GB)

In [ ]:
from google.colab import userdata
from huggingface_hub import login, snapshot_download
login(token=userdata.get("HF_TOKEN"))
snapshot_download("meta-llama/Llama-3.1-8B-Instruct",
                  allow_patterns=["*.json", "*.safetensors", "tokenizer*"])  # skips the 16 GB original/*.pth

## 5. LongGenBench data, pinned
SCOPE's data files and its full-cache (`ALLKV`) Llama-3.1-8B-Instruct predictions, both settings, fetched from one
pinned commit and checked against a SHA-256 each (`data/longgenbench_loader.py`). About 17 MB.

In [ ]:
%%bash
set -euo pipefail
cd /content/RecoverKv
python -m data.longgenbench_loader --out data/longgenbench

## 6. Preflight
Every check here guards something the run would otherwise do silently wrong, or refuse hours in: the transformers pin,
the softmax LSE that L-reuse needs from flash-attn, Inductor compiling on this box, the read gate having a call site,
the shipped configs being at the operating point, the tokenizer (one BOS, `<|eot_id|>` as EOS, as SCOPE ran it), and
the two reproduction checks: **SCOPE's prompts rebuilt byte for byte** and **SCOPE's published scores recovered from its
own predictions**. Then the CPU unit tests for the runner, prompt and metric. It must print `PREFLIGHT OK`.

In [ ]:
%%bash
set -euo pipefail
cd /content/RecoverKv
python - <<'EOF'
import re, subprocess, sys, torch, transformers, triton, flash_attn
print("python", sys.version.split()[0], "| torch", torch.__version__, "cuda", torch.version.cuda,
      "| triton", triton.__version__, "| transformers", transformers.__version__,
      "| flash_attn", flash_attn.__version__)
assert torch.__version__.startswith("2.6"), "torch must be 2.6.x"
from utils.cache_factory import assert_transformers_version_supported
assert_transformers_version_supported()                     # == 4.47.1
p = torch.cuda.get_device_properties(0)
print(f"gpu {p.name}  {p.total_memory / 2**30:.1f} GiB")
from flash_attn import flash_attn_func                       # L-reuse needs the LSE back
q = torch.randn(1, 128, 32, 128, device="cuda", dtype=torch.float16)
k = torch.randn(1, 128, 8, 128, device="cuda", dtype=torch.float16)
out = flash_attn_func(q, k, k, causal=True, return_attn_probs=True)
assert isinstance(out, tuple) and tuple(out[1].shape) == (1, 32, 128), "flash-attn returned no LSE"
f = torch.compile(lambda x: (x * 2).sin().sum(-1))             # the eviction is compiled-or-raise
f(torch.randn(64, 64, device="cuda")); torch.cuda.synchronize()

# The read gate has a production call site (CLAUDE.md: "It is wired. Check, don't assume.")
grep = subprocess.run("grep -rn 'fused_gate' --include='*.py' . | grep -v '^./tests/'", shell=True,
                      capture_output=True, text=True).stdout.splitlines()
calls = [l for l in grep if re.search(r"=\s*fused_gate\(", l)]
assert calls, "fused_gate has no non-test call site: the decode step would not be gated"
print("read gate call site:", calls[0].split(":")[0] + ":" + calls[0].split(":")[1])
r = subprocess.run([sys.executable, "scripts/check_operating_point.py"], capture_output=True, text=True)
assert r.returncode == 0, r.stdout[-3000:]
print("shipped configs at the operating point")

# The tokenizer SCOPE ran: one BOS, added by the tokenizer; EOS is <|eot_id|>.
from transformers import AutoTokenizer
from utils.prompting import encode_prompt
import data.longgenbench_loader as L
tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
ids = encode_prompt(tok, L.build_prompt("gsm8k", "x"), return_tensors="pt").input_ids[0].tolist()
assert ids[0] == tok.bos_token_id and ids[1] != tok.bos_token_id, ids[:4]
assert tok.eos_token == "<|eot_id|>" and tok.eos_token_id == 128009, (tok.eos_token, tok.eos_token_id)

# Reproduction: SCOPE's prompts, byte for byte, and SCOPE's published numbers from its own predictions.
from modules.evaluation.longgenbench_scoring import rescore_reference
D = "data/longgenbench"
for s in L.SETTINGS:
    for t in L.TASKS:
        ids_ = L.example_ids(D, s, t)
        ref = L.reference_by_id(D, s, t, ids_)             # raises if any prompt differs
        assert sorted(ref) == ids_, f"{s}/{t}: SCOPE ran a different example set"
        got, want = rescore_reference(D, s, t), L.published_reference_score(s, t)
        assert got == want, f"{s}/{t}: metric gives {got}, SCOPE publishes {want}"
        print(f"  {s} {t:5s} {len(ids_):2d} examples: prompts identical to SCOPE's; "
              f"SCOPE full-cache score {got} == published {want}")
EOF
LONGGENBENCH_DATA=data/longgenbench python -m pytest -q -p no:cacheprovider tests/test_longgenbench.py
echo "PREFLIGHT OK"

## 7. Mount Google Drive
Every part is copied here as soon as it finishes, and earlier verified parts are reused from here,
so nothing is lost if the runtime is recycled.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 8. Settings (the only cell you edit)
The cache knobs are your efficiency notebook's settings, which are the shipped operating point
(`utils.config.OPERATING_POINT`). The budget is always `CACHE_BUDGET * (prompt + max_new_tokens)`. That is the method's own
definition, the one LongBench and GSM8K use, and the same as `BUDGET_BASIS = "prefill_plus_gen"` there. There is no
`"context"` basis in a quality run. Because the budget is sized on `max_new_tokens`, an answer that stops early holds
more than `CACHE_BUDGET` of its own sequence; the table reports by how much.

* `SETTING`: `"8K"` (60 / 60 / 80 questions per prompt, up to 7,950 new tokens) or `"4K"` (30 / 30 / 40, 4,096).
* `NUM_EXAMPLES`: `None` runs every example SCOPE scores (21 / 53 / 15 at 8K). An int runs the first N of each task.
  Then the SCOPE reference is re-scored on the same N, so the comparison stays like for like.
* `CHUNK`: examples per part. Keep it fixed between reruns, or earlier parts no longer match and are not reused.
* `DROP_SCOPE_OOM`: SCOPE drops MMLU rows 21, 30, 31, 48 (its 80 GB A100 ran out of memory on them). Keep `True` to
  compare with its table.

In [ ]:
SETTING      = "8K"            # "8K" | "4K"
TASKS        = ["gsm8k", "mmlu", "csqa"]
NUM_EXAMPLES = None            # None = all (21 / 53 / 15 at 8K); int = first N per task
WINDOW_SIZE  = 8               # multiple of 4 when QUANT_RATIO > 0
NUM_SINK     = 5
LOCAL_WINDOW = 128             # int multiple of WINDOW_SIZE
QUANT_RATIO  = 0.70            # int2 share of the byte budget; 0 = fp16 only (no gate)
GATE_RATIO   = 0.25            # fraction of int2 windows read per step, (0, 1]
CACHE_BUDGET = 0.20            # fraction of (prompt + max_new_tokens) kept, in bytes
QUANT_MODE   = "bytes"
DTYPE        = "float16"
MODEL_ID     = "meta-llama/Llama-3.1-8B-Instruct"
CONTROL_ROW  = False           # True adds ours at quant_gate_ratio 1.0 (the gate's control arm)
CHUNK        = 8               # examples per part
DROP_SCOPE_OOM = True
REUSE_VERIFIED = True          # reuse earlier parts with identical settings/commit/GPU that verify
READ_FRACTION_TOL = 0.05       # warn if the realised read fraction is further than this from GATE_RATIO
REPRO_TOL    = 5.0             # warn if fullkv differs from SCOPE's ALLKV by more points than this

## 9. Helpers (run as-is)

In [ ]:
import json, os, shutil, subprocess, sys, time, importlib
from pathlib import Path
import numpy as np

REPO = "/content/RecoverKv"
PIN = "72d8e07a93b36536d9a47f07bebb40cea66fed32"
sys.path.insert(0, REPO)
import data.longgenbench_loader as L
import modules.evaluation.longgenbench_scoring as S
L, S = importlib.reload(L), importlib.reload(S)

PY = shutil.which("python") or sys.executable
DATA_DIR = f"{REPO}/data/longgenbench"
ENV = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONHASHSEED="0", CUBLAS_WORKSPACE_CONFIG=":4096:8",
           TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
OUT_ROOT = Path(f"{REPO}/outputs/lgb")
DRIVE_MOUNT = Path("/content/drive/MyDrive")
DRIVE_ROOT = DRIVE_MOUNT / "recoverkv_longgenbench"
CARD_DEFAULT = {"mu": 4, "v": 8, "t": 8, "vm": 4}       # the shipped gate card
EOT_ID = 128009                                          # <|eot_id|>, SCOPE's eos_token_id
SMOKE_NEW = 512
SEC_PER_TOKEN = {"fullkv": 0.030, "ours": 0.055, "ctrl_gate1.0": 0.060}   # priors; replaced by measurements


class PartFailed(Exception):
    pass


# ----------------------------------------------------------------- settings and plan
def check_knobs():
    assert SETTING in L.SETTINGS, f"SETTING must be one of {L.SETTINGS}"
    assert TASKS and all(t in L.TASKS for t in TASKS), f"TASKS must be a subset of {L.TASKS}"
    assert NUM_EXAMPLES is None or int(NUM_EXAMPLES) >= 1
    assert 0.0 <= QUANT_RATIO <= 1.0, "QUANT_RATIO in [0, 1]"
    assert 0.0 < GATE_RATIO <= 1.0, "GATE_RATIO in (0, 1]"
    assert 0.0 < CACHE_BUDGET <= 1.0, "CACHE_BUDGET in (0, 1]"
    assert QUANT_MODE in ("bytes", "tokens")
    assert DTYPE == "float16", "SCOPE's reference ran fp16"
    if QUANT_RATIO > 0:
        assert WINDOW_SIZE % 4 == 0, "WINDOW_SIZE must be a multiple of 4 when QUANT_RATIO > 0"
    assert isinstance(LOCAL_WINDOW, int) and LOCAL_WINDOW % WINDOW_SIZE == 0, \
        "LOCAL_WINDOW must be an int multiple of WINDOW_SIZE"
    assert int(CHUNK) >= 1
    assert _git_head() == PIN, f"{REPO} is at {_git_head()[:10]}, not the pinned {PIN[:10]}: run step 2 again"
    L.assert_supported_model(MODEL_ID)
    for t in TASKS:
        L.dataset_sha(DATA_DIR, SETTING, t)              # present and byte-identical to the pin


def arms():
    a = [("fullkv", None), ("ours", float(GATE_RATIO))]
    if CONTROL_ROW and QUANT_RATIO > 0 and GATE_RATIO < 1.0:
        a.append(("ctrl_gate1.0", 1.0))
    return a


def task_ids(task):
    ids = L.example_ids(DATA_DIR, SETTING, task, DROP_SCOPE_OOM)
    return ids[:int(NUM_EXAMPLES)] if NUM_EXAMPLES else ids


def plan():
    parts = []
    for task in TASKS:
        ids = task_ids(task)
        for c0 in range(0, len(ids), int(CHUNK)):
            c1 = min(c0 + int(CHUNK), len(ids))
            for arm, gate in arms():
                parts.append(dict(task=task, start=c0, end=c1, ids=ids[c0:c1], arm=arm, gate=gate))
    return parts


def op_tag():
    return (f"b{CACHE_BUDGET}_q{QUANT_RATIO}_w{WINDOW_SIZE}_s{NUM_SINK}_l{LOCAL_WINDOW}"
            f"_{QUANT_MODE}_{DTYPE}")


def part_dir(p, smoke=False):
    sub = f"fullkv_{DTYPE}" if p["arm"] == "fullkv" else f"{p['arm']}_g{p['gate']}_{op_tag()}"
    return OUT_ROOT / ("smoke_" + SETTING if smoke else SETTING) / sub / \
        f"drop{int(DROP_SCOPE_OOM)}" / f"{p['task']}_{p['start']:03d}-{p['end']:03d}"


def _git_head():
    r = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True)
    return r.stdout.strip() or "no_git"


def _gpu_name():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                           capture_output=True, text=True)
    except OSError:
        return None
    return r.stdout.strip().splitlines()[0] if r.stdout.strip() else None


def part_settings(p, max_new=None):
    # Exactly what run_settings.json records; reuse compares against the same dict.
    s = dict(setting=SETTING, task=p["task"], start=p["start"], end=p["end"], ids=p["ids"],
             arm=p["arm"], model=MODEL_ID, dtype=DTYPE, drop_scope_oom=DROP_SCOPE_OOM,
             max_new_tokens_override=max_new, reference_commit=L.SCOPE_COMMIT, commit=_git_head())
    if p["arm"] != "fullkv":
        s.update(cache_budget=CACHE_BUDGET, window_size=WINDOW_SIZE, num_sink_tokens=NUM_SINK,
                 local_window_size=LOCAL_WINDOW, quant_ratio=QUANT_RATIO,
                 quant_budget_mode=QUANT_MODE, quant_gate_ratio=p["gate"], first_eviction_step=0)
    return s


def config_text(p, out, max_new=None):
    j = json.dumps
    lines = ["# generated by notebooks/longgenbench_colab.ipynb",
             "run: {mode: longgenbench, seed: 42}",
             f"model: {{name: {j(MODEL_ID)}, revision: null, dtype: {DTYPE}, "
             "attn_implementation: flash_attention_2}"]
    if p["arm"] == "fullkv":
        lines += ["cache: {backend: dynamic, backend_package: null, cache_budget: null}"]
    else:
        lines += ["cache:",
                  "  backend: windowed",
                  "  backend_package: flash_attn",
                  f"  cache_budget: {float(CACHE_BUDGET)}",
                  f"  window_size: {int(WINDOW_SIZE)}",
                  f"  num_sink_tokens: {int(NUM_SINK)}",
                  f"  local_window_size: {int(LOCAL_WINDOW)}",
                  f"  quant_ratio: {float(QUANT_RATIO)}",
                  f"  quant_budget_mode: {QUANT_MODE}",
                  f"  quant_gate_ratio: {float(p['gate'])}",
                  "  first_eviction_step: 0",
                  f"window: {{window_size: {int(WINDOW_SIZE)}, num_sink_tokens: {int(NUM_SINK)}, "
                  f"local_window_size: {int(LOCAL_WINDOW)}}}"]
    lines += ["telemetry: {track_scores: false}",
              "longgenbench:",
              f"  data_dir: {j(DATA_DIR)}",
              f"  setting: {j(SETTING)}",
              f"  tasks: [{p['task']}]",
              f"  output_dir: {j(str(out))}",
              f"  start: {p['start']}",
              f"  end: {p['end']}",
              f"  drop_scope_oom: {str(bool(DROP_SCOPE_OOM)).lower()}",
              "  skip_oom: true",
              "  aggressive_cache_clear: true",
              f"  max_new_tokens: {'null' if max_new is None else int(max_new)}",
              ""]
    return "\n".join(lines)


# ----------------------------------------------------------------- files: always fresh
def _fresh(path):
    path = Path(path)
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()
    return path


def _drive_path(local):
    return DRIVE_ROOT / Path(local).relative_to(OUT_ROOT)


def _to_drive(src):
    if not DRIVE_MOUNT.is_dir():
        return
    src = Path(src)
    dst = _fresh(_drive_path(src))
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst) if src.is_dir() else shutil.copy2(src, dst)
    print("copied to", dst)


# ----------------------------------------------------------------- the cache geometry, before any GPU hour
GEO_PY = r"""
import json, sys, statistics as st, torch
from transformers import AutoConfig, AutoTokenizer
from utils.cache_factory import get_cache_classes
from utils.prompting import encode_prompt
import data.longgenbench_loader as L
a = json.loads(sys.argv[1])
tok = AutoTokenizer.from_pretrained(a["model"]); mc = AutoConfig.from_pretrained(a["model"])
_, WCC, _ = get_cache_classes("flash_attn")
cc = WCC(window_size=a["w"], num_sink_tokens=a["s"], local_window_size=a["l"], cache_budget=a["b"],
         quant_ratio=a["q"], quant_budget_mode=a["mode"], quant_gate_ratio=a["g"], first_eviction_step=0)
out = {}
for t in a["tasks"]:
    ex = L.load_longgenbench(a["data"], a["setting"], t, a["drop"])
    n = [len(encode_prompt(tok, e["prompt"], return_tensors="pt").input_ids[0]) for e in ex]
    mx = L.MAX_NEW_TOKENS[a["setting"]]
    rows = {}
    for name, P in (("min", min(n)), ("median", int(st.median(n))), ("max", max(n))):
        r = cc.resolve(P, mc, torch.float16, mx)
        rows[name] = dict(prompt=P, budget_tokens=r.total_budget_tokens, N_q=r.N_q, top_k_fp=r.top_k_fp,
                          fp_cap=r.num_sink_tokens + r.local_tokens + r.top_k_fp * r.window_size,
                          cap=r.retained_tokens, local=r.local_tokens, util=round(r.budget_utilisation, 3))
    out[t] = rows
print("GEO" + json.dumps(out))
"""
_GEO = {}


def geometry():
    key = (SETTING, tuple(TASKS), DROP_SCOPE_OOM, op_tag(), GATE_RATIO)
    if key in _GEO:
        return _GEO[key]
    arg = json.dumps(dict(model=MODEL_ID, w=WINDOW_SIZE, s=NUM_SINK, l=LOCAL_WINDOW, b=CACHE_BUDGET,
                          q=QUANT_RATIO, mode=QUANT_MODE, g=GATE_RATIO, tasks=TASKS, data=DATA_DIR,
                          setting=SETTING, drop=DROP_SCOPE_OOM))
    r = subprocess.run([PY, "-c", GEO_PY, arg], cwd=REPO, env=ENV, capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith("GEO")]
    if r.returncode or not line:
        raise SystemExit("cache geometry preview failed:\n" + r.stderr[-3000:])
    geo = json.loads(line[0][3:])
    print(f"cache geometry at your settings ({SETTING}, max_new_tokens {L.MAX_NEW_TOKENS[SETTING]}):")
    print(f"  {'task':6s} {'prompt':>14s} {'budget tok':>11s} {'fp16 keys':>10s} {'int2 win':>9s} "
          f"{'keys kept':>10s} {'bytes used':>10s}")
    bad = []
    for t, rows in geo.items():
        for name in ("min", "median", "max"):
            g = rows[name]
            print(f"  {t:6s} {name + ' ' + str(g['prompt']):>14s} {g['budget_tokens']:>11d} "
                  f"{g['fp_cap']:>10d} {g['N_q']:>9d} {g['cap']:>10d} {g['util']:>10.3f}")
        if QUANT_RATIO > 0 and rows["min"]["N_q"] == 0:
            bad.append(f"{t}: no int2 window fits the budget (N_q = 0): no int2 tier, nothing to gate")
        elif QUANT_RATIO > 0 and rows["min"]["prompt"] <= rows["min"]["fp_cap"]:
            print(f"  note: {t}'s shortest prompt fits in the fp16 tier; its int2 tier fills during decode")
    print("  (fp16 keys = sink + local + fp windows; a prompt longer than that is int2-compressed on decode step 0."
          "\n   keys kept = both tiers; past it, keys are dropped.)")
    if bad:
        raise SystemExit("\n".join(bad))
    _GEO[key] = geo
    return geo


# ----------------------------------------------------------------- one part = one run of main.py
def run_part(p, out, max_new=None):
    out = Path(out)
    _fresh(out)
    if DRIVE_MOUNT.is_dir():
        _fresh(_drive_path(out))
    out.mkdir(parents=True)
    cfg = out / "_longgenbench.generated.yaml"
    cfg.write_text(config_text(p, out, max_new))
    (out / "run_settings.json").write_text(json.dumps(part_settings(p, max_new), indent=2))
    t0 = time.time()
    tail = []
    with open(out / "run.log", "w") as log:
        proc = subprocess.Popen([PY, "main.py", "--config", str(cfg)], cwd=REPO, env=ENV,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            tail = (tail + [line])[-40:]
            if any(k in line for k in ("new tokens in", "read gate", "compression:", "never filled",
                                        "THE READ GATE", "Error", "error:", "Traceback", "WARNING")):
                print("   ", line.rstrip()[:220])
        rc = proc.wait()
    (out / "wall_s.txt").write_text(f"{time.time() - t0:.1f}\n")
    if rc:
        txt = "".join(tail)
        kind = "ran out of GPU memory" if any(k in txt.lower() for k in ("out of memory", "outofmemory")) \
            else f"main.py exited with {rc}"
        print(txt)
        raise PartFailed(f"{kind}; full log: {out}/run.log")
    return out


# ----------------------------------------------------------------- checks
def _load(p, out):
    task = p["task"]
    meta = json.loads((Path(out) / f"{task}.meta.json").read_text())
    recs = S.read_records(Path(out) / f"{task}.jsonl")
    return meta, recs


def verify(p, out, max_new=None, quiet=False):
    # Every field a sidecar must carry for its number to belong to this part, arm and setting.
    fails, warns = [], []
    try:
        meta, recs = _load(p, out)
    except (OSError, ValueError) as e:
        return False, [f"incomplete: {e}"], []

    def eq(k, want, src=None):
        got = (meta if src is None else src).get(k)
        if got != want:
            fails.append(f"{k} = {got!r}, expected {want!r}")

    eq("benchmark", "LongGenBench"); eq("setting", SETTING); eq("task", p["task"])
    eq("reference_commit", L.SCOPE_COMMIT)
    eq("dataset_sha", L.DATA_FILES[L.data_relpath(SETTING, p["task"])])
    eq("prompt_format", L.PROMPT_FORMAT); eq("drop_scope_oom", bool(DROP_SCOPE_OOM))
    eq("ids", p["ids"]); eq("max_new_tokens", int(max_new or L.MAX_NEW_TOKENS[SETTING]))
    eq("max_new_tokens_override", max_new)
    eq("model_name", MODEL_ID); eq("dtype", DTYPE); eq("attn_implementation", "flash_attention_2")
    eq("transformers_version", "4.47.1")
    if not str(meta.get("commit_sha") or "").startswith(PIN):
        fails.append(f"commit {str(meta.get('commit_sha'))[:10]} is not the pinned {PIN[:10]}")
    gen = meta.get("generation") or {}
    if (gen.get("eos_token_id") != [EOT_ID] or gen.get("num_beams") != 1 or gen.get("do_sample") is not False
            or gen.get("batch_size") != 1):
        fails.append(f"generation {gen} is not SCOPE's (greedy, B=1, eos=[{EOT_ID}])")
    if [r.get("id") for r in recs] != p["ids"]:
        fails.append("records do not cover exactly this part's examples")
    n_null = sum(r.get("pred") is None for r in recs)
    if n_null:
        warns.append(f"{n_null} example(s) ran out of GPU memory: excluded from every arm's paired numbers")
    rg = meta.get("read_gate") or {}
    if p["arm"] == "fullkv":
        eq("cache_type", "full_cache")
        if rg.get("verdict") != "not-expected":
            fails.append(f"full-cache read gate verdict {rg.get('verdict')!r}, expected 'not-expected'")
    else:
        eq("cache_type", "windowed"); eq("cache_backend_package", "flash_attn")
        for k, v in (("cache_budget", float(CACHE_BUDGET)), ("window_size", int(WINDOW_SIZE)),
                     ("num_sink_tokens", int(NUM_SINK)), ("local_window_size", int(LOCAL_WINDOW)),
                     ("quant_ratio", float(QUANT_RATIO)), ("quant_budget_mode", QUANT_MODE),
                     ("quant_gate_ratio", float(p["gate"])), ("first_eviction_step", 0),
                     ("quant_promotion", "bidir"), ("quant_promote_source", "dequant"),
                     ("quant_card_bits", CARD_DEFAULT)):
            eq(k, v)
        if (meta.get("quant_completion") or {}).get("mode") != "native":
            fails.append(f"quant_completion {meta.get('quant_completion')!r} is not the shipped kernel path")
        want_v = "gated" if QUANT_RATIO > 0 else "not-expected"
        if rg.get("verdict") != want_v:
            fails.append(f"read gate verdict {rg.get('verdict')!r}, expected {want_v!r} "
                         f"(gated={rg.get('gated')} fired={rg.get('fired')})")
        if want_v == "gated":
            if not rg.get("fired") or rg.get("gated") != rg.get("fired"):
                fails.append(f"gate ran on {rg.get('gated')} of {rg.get('fired')} fused layers")
            if rg.get("completion_applied"):
                fails.append("the completion-ablation epilogue ran: not the shipped method")
            rf = rg.get("read_fraction")
            if rf is None or abs(rf - float(p["gate"])) > READ_FRACTION_TOL:
                warns.append(f"realised read fraction {rf} vs configured {p['gate']}")
        geo = meta.get("resolved_geometry_first_example") or {}
        if QUANT_RATIO > 0 and not geo.get("N_q"):
            fails.append(f"NO int2 tier at this shape (N_q = {geo.get('N_q')}): nothing to gate")
        for k, v in (("window_size", int(WINDOW_SIZE)), ("num_sink_tokens", int(NUM_SINK)),
                     ("local_tokens", int(LOCAL_WINDOW)), ("quant_budget_mode", QUANT_MODE)):
            eq(k, v, geo)
        if meta.get("n_examples_no_int2"):
            warns.append(f"{meta['n_examples_no_int2']} example(s) never filled past the fp16 tier "
                         "(no int2 window, no gate on those)")
    if not quiet:
        for w in warns:
            print("  warn:", w)
        for f in fails:
            print("  FAIL:", f)
    return not fails, fails, warns


def report_part(p, out, max_new=None):
    meta, recs = _load(p, out)
    s = S.summarize(p["task"], recs)
    rg = meta.get("read_gate") or {}
    wall = float((Path(out) / "wall_s.txt").read_text()) if (Path(out) / "wall_s.txt").is_file() else None
    line = (f"  {p['task']} [{p['start']}, {p['end']}) {p['arm']}: acc {s['acc']:.2f}  aligned "
            f"{s['acc_aligned']:.2f}  coverage {s['coverage']:.1f}%  mean new tokens {s.get('mean_gen_tokens')}"
            f"  hit cap {s.get('pct_hit_token_cap')}%  peak {meta.get('peak_mem_gib_max')} GiB")
    if p["arm"] != "fullkv":
        rf = rg.get("read_fraction")
        line += (f"\n    gate {rg.get('verdict')} ({rg.get('gated')}/{rg.get('fired')} layers, read "
                 f"{'n/a' if rf is None else f'{rf:.3f}'})"
                 f"  int2 unused on {meta.get('n_examples_no_int2')}/{len(recs)}  no key dropped on "
                 f"{meta.get('n_examples_no_eviction')}/{len(recs)}  KV bytes <= "
                 f"{meta.get('mean_kv_bytes_fraction_upper')} of full")
    if wall:
        line += f"\n    part wall {wall / 60:.1f} min ({meta.get('total_gen_tokens')} new tokens)"
    print(line)
    ok, _, _ = verify(p, out, max_new)
    print("  VERIFIED" if ok else "  NOT VERIFIED: do not quote this part")
    return ok


def _restore(p, out):
    # A part that is on Drive but not on this disk (new runtime): bring it back.
    out = Path(out)
    if (not (out / f"{p['task']}.meta.json").is_file() and DRIVE_MOUNT.is_dir()
            and _drive_path(out).is_dir()):
        shutil.copytree(_drive_path(out), out, dirs_exist_ok=True)
        print("  found in Drive:", out.relative_to(OUT_ROOT))
    return out


def _reusable(p, out):
    if not REUSE_VERIFIED:
        return False
    out = _restore(p, out)
    try:
        st = json.loads((out / "run_settings.json").read_text())
    except (OSError, ValueError):
        return False
    want = part_settings(p)
    diff = {k: (st.get(k), v) for k, v in want.items() if st.get(k) != v}
    if diff:
        print("  earlier result not reusable, settings differ (was, now):", diff)
        return False
    try:
        meta, _ = _load(p, out)
    except (OSError, ValueError):
        print("  earlier result not reusable: incomplete")
        return False
    if meta.get("gpu_name") != _gpu_name():
        print(f"  earlier result not reusable: recorded on {meta.get('gpu_name')}, this GPU is {_gpu_name()}")
        return False
    ok, _, _ = verify(p, out)
    return ok


# ----------------------------------------------------------------- time projection
def _est_tokens(task, ids):
    ref = L.reference_by_id(DATA_DIR, SETTING, task, ids)
    cap = L.MAX_NEW_TOKENS[SETTING]
    return sum(min(len(ref[i]["pred"]) / 3.3, cap) if i in ref else cap for i in ids)


def _projection(remaining, measured):
    sec = 0.0
    for p in remaining:
        spt = measured.get(p["arm"]) or SEC_PER_TOKEN[p["arm"]]
        sec += _est_tokens(p["task"], p["ids"]) * spt + 90.0     # + model load and compiles
    return sec / 3600


# ----------------------------------------------------------------- the whole run
def run_all():
    check_knobs()
    geometry()
    parts = plan()
    print(f"\nGPU: {_gpu_name()} | commit: {_git_head()[:8]} | Drive: "
          f"{'mounted' if DRIVE_MOUNT.is_dir() else 'NOT mounted (results stay local)'}")
    print(f"{len(parts)} parts: " + ", ".join(f"{t} {len(task_ids(t))} examples" for t in TASKS)
          + f" x {len(arms())} arms; projected {_projection(parts, {}):.1f} h if nothing is reused")
    failed, measured, gen_tok, gen_sec = [], {}, {}, {}
    for k, p in enumerate(parts):
        out = part_dir(p)
        print(f"\n#################### part {k + 1}/{len(parts)}: {SETTING} {p['task']} "
              f"[{p['start']}, {p['end']}) {p['arm']} ####################")
        if _reusable(p, out):
            print("  reusing the verified earlier result, not re-run")
            continue
        try:
            run_part(p, out)
        except PartFailed as e:
            failed.append(f"{p['task']} [{p['start']}, {p['end']}) {p['arm']}: {e}")
            print(f"\n!! {failed[-1]}\n   flagged; continuing with the next part")
            _to_drive(out)
            continue
        if not report_part(p, out):
            failed.append(f"{p['task']} [{p['start']}, {p['end']}) {p['arm']}: did not verify")
        _to_drive(out)
        meta, _ = _load(p, out)
        gen_tok[p["arm"]] = gen_tok.get(p["arm"], 0) + int(meta.get("total_gen_tokens") or 0)
        gen_sec[p["arm"]] = gen_sec.get(p["arm"], 0.0) + float((out / "wall_s.txt").read_text())
        measured = {a: gen_sec[a] / gen_tok[a] for a in gen_tok if gen_tok[a]}
        print(f"  measured: " + ", ".join(f"{a} {1e3 * v:.0f} ms/token (part wall clock)"
                                          for a, v in measured.items())
              + f" | remaining ~{_projection(parts[k + 1:], measured):.1f} h")
    return final(failed)


# ----------------------------------------------------------------- the table
def _boot_ci(d, iters=10000, seed=0):
    d = np.asarray(d, float)
    if len(d) < 2:
        return float(d.mean()) if len(d) else float("nan"), float("nan"), float("nan")
    m = d[np.random.default_rng(seed).integers(0, len(d), size=(iters, len(d)))].mean(1)
    return float(d.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


def final(failed=()):
    check_knobs()
    rows, notes, unverified = [], list(failed), []
    summary = {"setting": SETTING, "operating_point": op_tag(), "gate_ratio": GATE_RATIO,
               "commit": _git_head(), "gpu": _gpu_name(), "tasks": {}}
    hdr = (f"{'task':6s} {'arm':13s} {'n':>3s} {'acc':>7s} {'aligned':>8s} {'cover%':>7s} "
           f"{'d acc vs fullkv [95% CI]':>26s} {'new tok':>8s} {'cap%':>5s} {'KVbytes<=':>9s} "
           f"{'read':>6s} {'tok/s*':>7s} {'peakGiB':>8s}")
    lines = [hdr, "-" * len(hdr)]
    task_acc = {}
    for task in TASKS:
        by_arm = {}
        for arm, gate in arms():
            recs, metas, missing = {}, [], 0
            for p in plan():
                if p["task"] != task or p["arm"] != arm:
                    continue
                out = _restore(p, part_dir(p))
                ok, fails, _ = verify(p, out, quiet=True) if out.is_dir() else (False, ["not run yet"], [])
                if not ok:
                    missing += len(p["ids"])
                    unverified.append(f"{task} [{p['start']}, {p['end']}) {arm}: {fails[0] if fails else '?'}")
                    continue
                meta, rs = _load(p, out)
                metas.append(meta)
                recs.update({r["id"]: r for r in rs})
            by_arm[arm] = (recs, metas, missing)
        ids = [i for i in task_ids(task)
               if all(i in by_arm[a][0] and by_arm[a][0][i].get("pred") is not None for a, _ in arms())]
        oom = sorted({i for a, _ in arms() for i, r in by_arm[a][0].items() if r.get("pred") is None})
        if oom:
            notes.append(f"{task}: example id(s) {oom} ran out of GPU memory in at least one arm and are "
                         "excluded from every arm")
        if not ids:
            lines.append(f"{task:6s} (no example has a verified result in every arm yet)")
            continue
        ref = L.reference_by_id(DATA_DIR, SETTING, task, ids)
        ref_acc = S.scope_task_score(task, [ref[i]["pred"] for i in ids], [ref[i]["answers"] for i in ids])
        ref_s = S.summarize(task, [dict(pred=ref[i]["pred"], answers=ref[i]["answers"]) for i in ids])
        lines.append(f"{task:6s} {'SCOPE ALLKV':13s} {len(ids):>3d} {ref_acc:>7.2f} {ref_s['acc_aligned']:>8.2f} "
                     f"{ref_s['coverage']:>7.1f}   (published full-cache predictions, same examples)")
        per_ex = {}
        tsum = {"n": len(ids), "scope_allkv": ref_acc}
        for arm, gate in arms():
            recs, metas, missing = by_arm[arm]
            rs = [recs[i] for i in ids]
            s = S.summarize(task, rs)
            per_ex[arm] = np.array([100 * S.scope_example_accuracy(task, r["pred"], r["answers"]) for r in rs])
            if arm == "fullkv":
                d_txt = ""
            else:
                m, lo, hi = _boot_ci(per_ex[arm] - per_ex["fullkv"])
                d_txt = f"{m:+.2f} [{lo:+.2f}, {hi:+.2f}]"
            tg = sum(int(mm.get("total_gen_tokens") or 0) for mm in metas)
            tw = sum(float(mm.get("total_generate_wall_s") or 0) for mm in metas)
            kvb = [r.get("kv_bytes_fraction_upper") for r in rs if r.get("kv_bytes_fraction_upper") is not None]
            rd = [mm.get("read_gate", {}).get("read_fraction") for mm in metas]
            rd = [x for x in rd if x is not None]
            peak = max((float(mm.get("peak_mem_gib_max") or 0) for mm in metas), default=float("nan"))
            rd_txt = f"{np.mean(rd):.3f}" if rd else "-"
            tps_txt = f"{tg / tw:.1f}" if tw else "-"
            lines.append(f"{task:6s} {arm:13s} {len(ids):>3d} {s['acc']:>7.2f} {s['acc_aligned']:>8.2f} "
                         f"{s['coverage']:>7.1f} {d_txt:>26s} {s.get('mean_gen_tokens', 0):>8.0f} "
                         f"{s.get('pct_hit_token_cap', 0):>5.1f} "
                         f"{(np.mean(kvb) if kvb else 1.0):>9.3f} {rd_txt:>6s} {tps_txt:>7s} {peak:>8.2f}")
            tsum[arm] = dict(acc=s["acc"], acc_aligned=s["acc_aligned"], coverage=s["coverage"],
                             mean_gen_tokens=s.get("mean_gen_tokens"), delta_vs_fullkv=d_txt or None,
                             kv_bytes_fraction_upper=(float(np.mean(kvb)) if kvb else 1.0),
                             read_fraction=(float(np.mean(rd)) if rd else None))
            task_acc.setdefault(arm, []).append(s["acc"])
        task_acc.setdefault("SCOPE ALLKV", []).append(ref_acc)
        # Does fullkv reproduce SCOPE's full cache? Score, and how long the two texts agree.
        fk = by_arm["fullkv"][0]
        pref = [len(os.path.commonprefix([fk[i]["pred"], ref[i]["pred"]])) for i in ids]
        same = sum(fk[i]["pred"] == ref[i]["pred"] for i in ids)
        d_ref = round(float(np.mean(per_ex["fullkv"])) - ref_acc, 2) + 0.0
        notes.append(f"{task}: fullkv - SCOPE ALLKV = {d_ref:+.2f} points; identical generations "
                     f"{same}/{len(ids)}; texts agree for a median {int(np.median(pref))} characters "
                     f"(min {min(pref)})")
        if abs(d_ref) > REPRO_TOL:
            notes.append(f"  !! {task}: fullkv does NOT reproduce SCOPE's full cache within {REPRO_TOL} points")
        if np.median(pref) < 200:
            notes.append(f"  !! {task}: fullkv and SCOPE diverge within 200 characters; check the prompt and stack")
        tsum["fullkv_vs_scope"] = dict(delta=d_ref, identical=same, median_prefix_chars=int(np.median(pref)))
        summary["tasks"][task] = tsum
        lines.append("")
    if len(task_acc.get("fullkv", [])) == len(TASKS) and len(TASKS) > 1:
        for arm in ["SCOPE ALLKV"] + [a for a, _ in arms()]:
            lines.append(f"{'avg':6s} {arm:13s} {'':>3s} {np.mean(task_acc[arm]):>7.2f}   (mean of task scores)")
    lines.append("\n* tok/s is wall clock inside a quality run (prefill, compiles and Python included): context, "
                 "not a performance number.\n  KVbytes<= is an upper bound on the KV bytes held at the end of each "
                 "example, over the full cache's, averaged.")
    lines.append("\nfull-cache reproduction, and exclusions:")
    lines += ["  " + n for n in notes]
    if unverified:
        lines.append("\nNOT in the table (missing or did not verify; rerun step 11):")
        lines += ["  " + u for u in unverified]
    table = "\n".join(lines)
    path = _fresh(OUT_ROOT / f"{SETTING}_FINAL_{op_tag()}_g{GATE_RATIO}_drop{int(DROP_SCOPE_OOM)}.txt")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(table + "\n")
    path.with_suffix(".json").write_text(json.dumps(summary, indent=2, default=str) + "\n")
    print("\n=================== FINAL TABLE (SCOPE metric, paired examples) ===================")
    print(table)
    ok = not unverified and not failed
    print("\nEVERY PART RAN AND VERIFIED." if ok else
          "\nNOT EVERY PART VERIFIED. Run step 11 again: verified parts are reused, only the others re-run.")
    print("written to", path)
    _to_drive(path)
    _to_drive(path.with_suffix(".json"))
    return ok


check_knobs()
print("helpers loaded | parts:", len(plan()), "| GPU:", _gpu_name(), "| commit:", _git_head()[:8])

## 10. Smoke run (~10 min). Must print `SMOKE OK`
Before any GPU hour: the cache geometry at your settings for every task's prompts (it refuses a setting with no int2 tier).
Then one GSM8K example per arm, capped at 512 new tokens, through the same code path as step 11. It checks the plumbing,
not the score: every sidecar field, the gate verdict, that the int2 tier was used, and how far fullkv's text agrees with
SCOPE's own full-cache output for the same prompt.

In [ ]:
check_knobs()
geometry()
first = task_ids("gsm8k")[:1]
ok_all, smoke_out = True, {}
for arm, gate in arms():
    p = dict(task="gsm8k", start=0, end=1, ids=first, arm=arm, gate=gate)
    out = part_dir(p, smoke=True)
    print(f"\n---- smoke: {arm} ----")
    run_part(p, out, max_new=SMOKE_NEW)
    ok_all &= report_part(p, out, max_new=SMOKE_NEW)
    smoke_out[arm] = _load(p, out)[1][0]
ref = L.reference_by_id(DATA_DIR, SETTING, "gsm8k", first)[first[0]]["pred"]
n = len(os.path.commonprefix([smoke_out["fullkv"]["pred"], ref]))
print(f"\nfullkv vs SCOPE's full-cache text for the same prompt: identical for the first {n} characters "
      f"(of {len(smoke_out['fullkv']['pred'])} generated here)")
if n < 200:
    print("  !! that is short. A greedy run on the same prompt should agree for a long stretch; a near-tie can still "
          "flip it early on a different stack. If the full run's median is also short, stop and investigate.")
assert ok_all, "smoke run did not verify: do not start step 11"
print("\nSMOKE OK")

## 11. The real run
For each task and each chunk of examples, fullkv runs first, then ours (then the control, if on). Each part is printed,
verified and copied to Drive before the next one starts, and the remaining time is re-projected from measured speed.
A part that fails a check or runs out of GPU memory is flagged, and the next part runs. Run this cell again at any
point: parts that already verified are reused, and only the others run.

Expected and harmless in the output: `operation scheduled before its operands`, `tl.where with a non-boolean
condition`, the TF32 note, `failed during evaluate_expr(u0 <= 0 ...)`, and transformers' notes that
`temperature`/`top_p` are ignored when `do_sample=False`.

In [ ]:
OK = run_all()

## 12. Reprint the final table
After a session restart, run steps 7–9 first. On a brand-new runtime, run steps 1–5 and 7–9: the parts come back from
Drive, and nothing is re-run. Only examples with a verified result in every arm enter the table.

In [ ]:
final()